# EDA — Sberbank Russian Housing Market

Разведка данных с использованием пайплайна из `src/`: загрузка train/test/macro, базовая статистика, распределение целевой, пропуски, ключевые признаки.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Корень проекта — для импорта src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, TARGET_COL, ID_COL, TIMESTAMP_COL
from src.data import load_and_merge, load_train, load_test, load_macro

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

## 1. Загрузка данных через пайплайн

Используем `load_and_merge()` из `src.data`: train и test объединяются с macro по дате сделки.

In [2]:
train, test = load_and_merge(RAW_DIR, with_macro=True)
print("Train:", train.shape)
print("Test:", test.shape)
print("Колонок в train (с macro):", train.shape[1])

Train: (30471, 391)
Test: (7662, 390)
Колонок в train (с macro): 391


In [ ]:
train.head(3)

## 2. Целевая переменная `price_doc`

Распределение цены и лог-трансформация (часто используется при обучении).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train[TARGET_COL].hist(bins=80, ax=axes[0], edgecolor="black", alpha=0.8)
axes[0].set_title("Распределение price_doc (руб)")
axes[0].set_xlabel("price_doc")

np.log1p(train[TARGET_COL]).hist(bins=80, ax=axes[1], edgecolor="black", alpha=0.8)
axes[1].set_title("Распределение log1p(price_doc)")
axes[1].set_xlabel("log1p(price_doc)")
plt.tight_layout()
plt.show()

In [ ]:
print("Базовая статистика price_doc:")
train[TARGET_COL].describe()

## 3. Временной диапазон и динамика цен

Даты сделок и медианная цена по времени (месяц).

In [ ]:
train[TIMESTAMP_COL] = pd.to_datetime(train[TIMESTAMP_COL])
test[TIMESTAMP_COL] = pd.to_datetime(test[TIMESTAMP_COL])
print("Train: с", train[TIMESTAMP_COL].min(), "по", train[TIMESTAMP_COL].max())
print("Test:  с", test[TIMESTAMP_COL].min(), "по", test[TIMESTAMP_COL].max())

In [ ]:
monthly = train.groupby(train[TIMESTAMP_COL].dt.to_period("M"))[TARGET_COL].median()
monthly.plot(title="Медианная цена сделки по месяцам")
plt.xlabel("Месяц")
plt.ylabel("price_doc (медиана)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Пропуски

Доля пропусков по колонкам (топ по пропускам).

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)
missing_top = missing[missing > 0].head(40)
missing_top.plot(kind="barh", figsize=(8, 10))
plt.title("Доля пропусков по признакам (топ-40)")
plt.xlabel("Доля NA")
plt.tight_layout()
plt.show()

In [ ]:
print("Колонок с хотя бы одним пропуском:", (missing > 0).sum())
print("Колонок без пропусков:", (missing == 0).sum())

## 5. Ключевые признаки: площадь, этаж, район, тип

Распределения и связь с ценой.

In [ ]:
key_cols = ["full_sq", "life_sq", "floor", "num_room", "build_year", "kitch_sq"]
existing = [c for c in key_cols if c in train.columns]
train[existing].describe()

In [ ]:
if "product_type" in train.columns:
    train["product_type"].value_counts().plot(kind="bar", title="Тип сделки")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
if "sub_area" in train.columns:
    areas = train["sub_area"].value_counts()
    print("Число уникальных районов (sub_area):", areas.shape[0])
    print("Топ-15 по количеству сделок:")
    display(areas.head(15))

In [ ]:
# Медианная цена по району (топ-20 по числу сделок)
if "sub_area" in train.columns:
    top_areas = train["sub_area"].value_counts().head(20).index
    by_area = train[train["sub_area"].isin(top_areas)].groupby("sub_area")[TARGET_COL].median().sort_values(ascending=True)
    by_area.plot(kind="barh", figsize=(8, 6), title="Медианная цена по району (топ-20 по числу сделок)")
    plt.xlabel("price_doc (медиана)")
    plt.tight_layout()
    plt.show()

## 6. Связь числовых признаков с ценой

Корреляции с `price_doc` и с `log1p(price_doc)`; scatter по площади.

In [ ]:
numeric = train.select_dtypes(include=[np.number])
if TARGET_COL in numeric.columns:
    corr_target = numeric.corr()[TARGET_COL].drop(TARGET_COL, errors="ignore").abs().sort_values(ascending=False)
    corr_target.head(20).plot(kind="barh", figsize=(8, 6), title="Корреляция с price_doc (топ-20)")
    plt.tight_layout()
    plt.show()

In [ ]:
if "full_sq" in train.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sample = train.sample(min(3000, len(train)), random_state=42)
    ax.scatter(sample["full_sq"], sample[TARGET_COL], alpha=0.3, s=10)
    ax.set_xlabel("full_sq")
    ax.set_ylabel("price_doc")
    ax.set_title("full_sq vs price_doc (sample)")
    plt.tight_layout()
    plt.show()

## 7. Подготовка фичей через пайплайн

Используем `prepare_features()` из `src.features`: производные признаки (дата, площади), кодирование, заполнение пропусков. Так мы видим, какие признаки пойдут в модель.

In [ ]:
from src.features import prepare_features

X_train, X_test, feature_names = prepare_features(
    train, test,
    add_derived=True,
    fill_na_numeric="median",
    categorical_strategy="ordinal",
    drop_high_missing=0.9,
)
print("Число признаков после подготовки:", len(feature_names))
print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

In [ ]:
print("Примеры признаков:", feature_names[:25])